# Model Training: Volatility LSTM
**Objective:** Train the Multivariate LSTM to predict log realized volatility. We will use the QLIKE loss function, which is mathematically superior for heteroscedastic volatility forecasting compared to MSE.

In [ ]:
import torch
import pandas as pd
import numpy as np
import os
import sys
from torch.utils.data import DataLoader, TensorDataset

# Add src to path
sys.path.append(os.path.abspath('../src'))
from models.vol_lstm import VolatilityLSTM, QLIKELoss

# Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

## 1. Data Preparation: Creating Rolling Window Tensors
The LSTM requires 3D input: `(Batch_Size, Sequence_Length, Num_Features)`. We convert our `train_engineered.csv` into this format using a rolling window of 78 bars (representing ~1 trading day).

In [ ]:
def create_sequences(df, features, target, seq_length=78):
    X, y = [], []
    data_feat = df[features].values
    data_target = df[target].values
    
    for i in range(len(data_feat) - seq_length):
        X.append(data_feat[i:i+seq_length])
        y.append(data_target[i+seq_length])
    return torch.tensor(np.array(X), dtype=torch.float32), torch.tensor(np.array(y), dtype=torch.float32)

# Load data
train_df = pd.read_csv('../data/processed/train_engineered.csv', index_col=0)
val_df = pd.read_csv('../data/processed/val_engineered.csv', index_col=0)

# Define features based on Model Strategy spec
FEATURES = ['log_return', 'log_volume_usdt', 'rv_1h', 'rv_1d', 'rv_1w', 'garman_klass_vol', 
            'rsi_14', 'ema_ratio', 'avg_trade_size', 'volume_z_score', 'amihud_illiquidity', 
            'vol_imbalance', 'vol_to_vol_ratio', 'sin_day', 'cos_day']
TARGET = 'target_log_rv_next'

X_train, y_train = create_sequences(train_df, FEATURES, TARGET)
X_val, y_val = create_sequences(val_df, FEATURES, TARGET)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=64, shuffle=False)

## 2. Model Training Loop
We initialize the `VolatilityLSTM` with a hidden size of 32 to avoid overfitting, as financial data has a low Signal-to-Noise Ratio (SNR).

In [ ]:
model = VolatilityLSTM(input_size=len(FEATURES), hidden_size=32, num_layers=2).to(device)
criterion = QLIKELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 20
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        output = model(X_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{epochs} | QLIKE Loss: {total_loss/len(train_loader):.6f}")

# Save the best weights
os.makedirs('../weights', exist_ok=True)
torch.save(model.state_dict(), '../weights/vol_lstm_best.pth')

## 3. Validation Performance
Finally, we check the band coverage. If the model is valid, $\sim95\%$ of actual returns should fall within the predicted dynamic volatility bands.

model.eval()
all_preds = []
with torch.no_grad():
    for X_batch, _ in val_loader:
        all_preds.append(model(X_batch.to(device)).cpu())
        
preds = torch.cat(all_preds).numpy()
# Add logic here to compare preds vs actuals and calculate Band Coverage
print("Training complete. Weights saved to ../weights/vol_lstm_best.pth")